# Assignment 2: AI-Assisted Data Cleaning & Fraud Detection

You can run this notebook in Google Colab or Jupyter Notebook.  
Please **read each instruction carefully** and complete each task step by step.

---

## Background

Online payment fraud is one of the most significant challenges facing the financial industry today. Every year, billions of dollars are lost to fraudulent transactions, making accurate and scalable fraud detection systems critically important.

This assignment is based on the **IEEE-CIS Fraud Detection** dataset, originally published as a Kaggle competition by the **IEEE Computational Intelligence Society (IEEE-CIS)** in collaboration with **Vesta Corporation**, a leading payment service company. The dataset contains real-world anonymized e-commerce transaction records, and the task is to predict whether a given transaction is fraudulent.

---

## Learning Goals

By working through this notebook, you will:

- Use a **Large Language Model (LLM)** to semantically understand and clean structured data.
- Translate LLM-generated cleaning logic into **executable Python/Pandas code**.
- Apply **two machine learning methods** to a real-world binary classification problem.
- Evaluate model performance using appropriate metrics for **imbalanced datasets**.

---

## Dataset Overview

This assignment uses the following data files:

### Task 1 — Raw Data (for LLM Cleaning)

| File | Rows | Description |
|------|------|-------------|
| `train_transaction.csv` | ~590,000 | Core transaction records with labels |
| `train_identity.csv` | ~144,000 | Identity information linked to transactions |

> **Note:** Not every transaction has a corresponding identity record. The join is a **left join** on `TransactionID`. In Task 1, you will only work with the **first 50 rows** of these files.

### Task 2 — Pre-processed Data (for Model Training)

| File | Rows | Description |
|------|------|-------------|
| `training_data.csv` | ~472,000 | Fully pre-processed training set (merged + cleaned) |
| `test_data.csv` | ~118,000 | Fully pre-processed test set (no labels) |
| `sample_submission.csv` | ~118,000 | Template for your prediction submission |

> **Note:** Not every transaction has a corresponding identity record. The join is a **left join** on `TransactionID`.

---

## Data Dictionary

### Transaction Table (~394 columns)

| Column(s) | Type | Description |
|-----------|------|-------------|
| `TransactionID` | int | Unique identifier for each transaction |
| `isFraud` | int (0/1) | **Target variable** — 1 = fraudulent, 0 = legitimate *(train only)* |
| `TransactionDT` | int | Timedelta (in seconds) from a reference point, **not** an actual timestamp |
| `TransactionAmt` | float | Transaction amount in USD |
| `ProductCD` | str | Product code: W, H, C, S, or R |
| `card1`–`card6` | int/str | Payment card information (type, bank, country, etc.) |
| `addr1`, `addr2` | int | Billing address (region, country) |
| `dist1`, `dist2` | float | Distance between addresses |
| `P_emaildomain` | str | Purchaser's email domain (e.g., gmail.com) |
| `R_emaildomain` | str | Recipient's email domain |
| `C1`–`C14` | float | Anonymized count features (e.g., number of addresses linked to a card) |
| `D1`–`D15` | float | Anonymized timedelta features (e.g., days since last transaction) |
| `M1`–`M9` | str (T/F) | Match features (e.g., whether name on card matches address) |
| `V1`–`V339` | float | Anonymized Vesta engineered features |

### Identity Table (~41 columns)

| Column(s) | Type | Description |
|-----------|------|-------------|
| `TransactionID` | int | Foreign key linking to the transaction table |
| `id_01`–`id_38` | int/str | Anonymized identity features (device, network, etc.) |
| `DeviceType` | str | `mobile` or `desktop` |
| `DeviceInfo` | str | Device model/OS string (e.g., `Windows`, `iOS 11.3`) |

---

## A Note on Class Imbalance

In this dataset, only about **3.5% of transactions are fraudulent**. This means a naive model that predicts "not fraud" for every transaction would achieve ~96.5% accuracy — but would be completely useless in practice.

For this reason, you should **not** use accuracy as your primary evaluation metric. Instead, use **AUC-ROC (Area Under the ROC Curve)**, recall and F1-score, which measures a model's ability to distinguish between the two classes regardless of the class distribution.

**Do remember, even your model marks a false fraud, the card holder can use OTP (one time password) by SMS to approve this transcation. But if you miss a true fraud, bank will lose real money.**

---

## Assignment Structure

| Task | Description |
|------|-------------|
| **Task 1.1** | Use an LLM to semantically clean and standardize selected columns from 50 rows of data |
| **Task 1.2** | With AI assistance, write a Python function that applies the same cleaning logic to the full dataset |
| **Task 2** | Train and evaluate two machine learning models on the pre-cleaned full dataset |

## You should deliver:

- This Notebook itself
- `Task1_deliver.csv`, generated by this Notebook
- `Task2_deliver.csv`, generated by this Notebook

## Special Note

You may need to use API key to leverage LLM. Your unique API key should be sent by Mr.LIU Muyun via email. If you haven't received it, please mail him (muyunliu@cuhk.edu.hk) to get it.

Those keys are properties of CUHK. **Please DONNOT leak the API key to anyone else.** 

## Preparation of libraries: Please run the following cell directly.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

# ── Data Handling ─────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocessing ─────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay
)

# ── Machine Learning ──────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
# Add or remove models above depending on your choice in Task 2

# ── LLM API (Task 1.1 & 1.2) ─────────────────────────────────────
from openai import OpenAI

print('All packages loaded successfully.')

---

## Task 1.1 — LLM-Assisted Semantic Data Cleaning

### Overview

Traditional data cleaning relies on manually written rules: replace missing values with the mean, drop outliers beyond 3 standard deviations, encode categories with a fixed mapping, and so on. These rules are effective, but they require the programmer to already understand the data.

**Large Language Models offer a different approach.** Because LLMs have been trained on vast amounts of text — including documentation, research papers, and domain knowledge — they can *reason about the meaning* of a column, not just its statistics. Given a column named `P_emaildomain` containing values like `gmail.com`, `googlemail.com`, and `gmai.com`, an LLM can recognize that the first two are the same provider and the third is likely a typo, and suggest a normalization strategy accordingly.

In this task, you will act as a **data analyst working with an LLM as your assistant**. Your goal is not to write cleaning code yet — it is to use the LLM to *understand* the data and *decide* what cleaning is needed.

---

### Columns to Clean

You will work with the following **6 columns** from `train_transaction.csv`. These columns were selected because they contain human-readable, semantically meaningful values that an LLM can reason about:

| Column | Example Values | Potential Issues |
|--------|---------------|-----------------|
| `ProductCD` | `W`, `H`, `C`, `S`, `R` | What do these codes mean? Are there invalid values? |
| `card4` | `visa`, `mastercard`, `american express` | Inconsistent capitalization, abbreviations |
| `card6` | `debit`, `credit`, `debit or credit` | Ambiguous categories, needs standardization |
| `P_emaildomain` | `gmail.com`, `googlemail.com`, `gmai.com` | Typos, aliases for the same provider |
| `M1`–`M9` | `T`, `F`, `NaN` | What do T/F mean in this context? How to handle missing values? |
| `DeviceInfo` | `Windows`, `iOS 11.3.0`, `SM-G935F Build/MMB29K` | Noisy strings, needs brand/OS extraction |

---

**Step 1 — Load the first 50 rows**

Load the first 50 rows of `train_transaction.csv` and extract the 6 columns listed above.

In [ ]:
target_cols_trans = ['TransactionID', 'ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1']
target_cols_ident = ['TransactionID', 'DeviceInfo']

df_trans = pd.read_csv('train_transaction.csv', usecols=target_cols_trans, nrows=50)
df_ident = pd.read_csv('train_identity.csv', usecols=target_cols_ident)

df_sample = df_trans.merge(df_ident, on='TransactionID', how='left')

df_sample.head(10)
df_sample.info()

**Step 2 — Explore the Data**

Before constructing your prompt, explore the 6 target columns to understand their unique values and missing counts. This will help you write a more precise prompt in Step 3.

In [ ]:
# Preview unique values and missing counts for each target column


**Step 3 — Ask the LLM to Clean the Data and Save the Result**

Now you will send both the column descriptions **and the raw 50-row data** to the LLM, and ask it to return the cleaned data directly in JSON format.

Your code should:
1. Construct a prompt that includes the raw data and asks the LLM to clean and return it as structured JSON
2. Call the API and parse the JSON response
3. Save the result as `Task1_deliver.csv`

> 💡 **Tip:** Ask the LLM to return a JSON array where each element is a cleaned row. Specify exactly what cleaning you expect for each column (e.g. normalize email aliases, extract device category, encode M1 as 0/1/-1).

Your deliverables for this task are:
1. The prompt you used
2. The saved `Task1_deliver.csv` file

In [ ]:
API_KEY = "Replace your unique API key here"


# The part is the API key calling.
# DONNOT change this part by yourself.
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.bianxie.ai/v1"
)

raw_data_json = df_sample.drop(columns=['TransactionID']).to_json(orient='records', indent=2)
# Pre-written part ends.

# You should start from here

prompt_clean = f"""
Your Prompt
"""

# Your part ends here.

# DONNOT change the following part.
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a data cleaning expert. Return only valid JSON, no explanation."},
        {"role": "user", "content": prompt_clean}
    ],
    temperature=0.0,
    max_tokens=4096
)

raw_output = response.choices[0].message.content

cleaned_data = json.loads(raw_output)
df_deliver = pd.DataFrame(cleaned_data)
df_deliver.to_csv('Task1_deliver.csv', index=False)

print("Saved Task1_deliver.csv")
print(df_deliver.head(10))

---

## Task 1.2 — AI-Assisted Code Generation for Full-Scale Cleaning

### Overview

In Task 1.1, you used an LLM to directly process a small sample of data row by row. However, this approach does not scale — a real dataset with hundreds of thousands of rows cannot fit into a single prompt.

The practical workflow is different: instead of asking the LLM to *process* the data, you ask it to *write the code* that processes the data. You then review and refine the generated code to ensure it correctly captures your cleaning logic.

In this task, you will use an LLM to generate a Python cleaning function based on the same logic you applied in Task 1.1.

> **Note:** You do **not** need to run any code in this task. Write your prompt and paste the generated function as instructed. Your submission will be reviewed for logical correctness and consistency with your Task 1.1 decisions.

---


**Step 1 — Construct a prompt asking the LLM to write a cleaning function**

Your prompt should:
- Describe each of the 6 columns and their cleaning requirements
- Be consistent with the cleaning decisions you made in Task 1.1
- Ask the LLM to write a single Python function `clean_dataframe(df)` that accepts a raw DataFrame and returns a cleaned copy

Paste your prompt in the markdown cell below.

**My Prompt (Deliverable 1):**

*(Paste your prompt here)*

**Step 2 — Paste and review the generated function**

Copy the LLM's output into the code cell below. Read through the logic carefully:
- Does it correctly handle missing values?
- Is it consistent with what you did in Task 1.1?
- Are there any errors or missing steps?

Make any corrections you think are necessary directly in the code cell.

In [ ]:
# Paste the codes you asked AI to generate.


---

## Task 2 — Machine Learning for Fraud Detection

### Overview

With the data now cleaned and standardized, you are ready to build predictive models. In this task, you will train **two machine learning classifiers** on the full training dataset and evaluate their ability to detect fraudulent transactions.

Fraud detection is a classic **binary classification** problem, but with a critical challenge: the dataset is highly **imbalanced** — only about 3.5% of transactions are fraudulent. This means that standard accuracy is a misleading metric. A model that blindly predicts "not fraud" for every transaction would achieve over 96% accuracy while catching zero fraud cases.

To address this, you will evaluate your models using **AUC-ROC** and **F1-score**, which are robust to class imbalance and better reflect real-world utility.


### Part 1 — Model Training
Load `training_data.csv`, inspect it, then train a classifier to predict `isFraud`.
Your deliverables for Part 1:
- A trained model
- AUC-ROC and F1-score on a held-out validation split
- A ROC curve and confusion matrix


In [ ]:
# This part has been written for you.
# However, if you use Colab, you may change this path to the mounted one.

df_train = pd.read_csv('training_data.csv')

print(f"Shape: {df_train.shape}")
print(f"Fraud rate: {df_train['isFraud'].mean():.4f}")
print(f"\nColumn types:\n{df_train.dtypes.value_counts()}")
print(f"\nMissing values: {df_train.isnull().sum().sum()}")
df_train.head(3)

In [ ]:
# Your Fisrt Model

In [ ]:
# Your Second Model

Please explain you choose which model and why here.

---

### Part 2 — Prediction on Test Set

Load `test_data.csv`, apply the same preprocessing, and generate fraud probability predictions for each transaction.

Submit your predictions as a CSV file named `Task2_deliver.csv` following the format of `sample_submission.csv`:

| Column | Description |
|--------|-------------|
| `TransactionID` | Transaction identifier |
| `isFraud` | Predicted fraud probability (float between 0 and 1) |

In [ ]:
# Use the model you trained in Part 1 to make predictions on the test set.
# Save your predictions as a CSV file named `Task2_deliver.csv` following the format of `sample_submission.csv`.

**End of the assignment 2**